In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

similarity_model = SentenceTransformer("all-MiniLM-L6-v2")

START_SIM = 0.90
MIN_SIM = 0.40
STEP = 0.05
MAX_SIM = 0.9999
MAX_RETRIES = 10

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [2]:
import json
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

INPUT_FILE = "/kaggle/input/datasets/debayushdey/contamination-study/mmlu_200_original.json"
OUTPUT_FILE = "/kaggle/working/paraphrased_200.json"

with open(INPUT_FILE, "r") as f:
    data = json.load(f)

print("Loaded:", len(data))

Loaded: 200


In [3]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [4]:
def semantic_similarity(q1, q2):

    emb1 = similarity_model.encode([q1])
    emb2 = similarity_model.encode([q2])

    sim = cosine_similarity(emb1, emb2)[0][0]

    return float(sim)

In [5]:
def paraphrase_question(question, choices):

    user_prompt = f"""
MOST IMPORTANT:"PLEASE PLEASE PLEASE DO NOT WRITE QUESTIONS EXACTLY SAME MAKE AT LEAST A COUPLE OF WORD 
CHANGES IF NOTHING ELSE IS FEASIBLE WHILE NOT ALTERING MEANING.ALWAYS AT LEAST CHANGE A COUPLE OF WORD PLEASE.PLEASE
DO NOT WRITE THE WORDS 'REWRITTEN QUESTION' OR 'QUESTION' IN YOUR GENERATED PARAPHRASE"
Rewrite the following multiple choice question with wording changes.
Make changes as you see fit but try to not change the meaning.Make minimal changes
at least in every question. Dont rewrite a question exact word for word. Just change a 
couple of words here and there while preserving the meaning.
ALWAYS REMEMBER:DO NOT GIVE THE SAME QUESTION AS THE PARAPHRASED QUESTION WITHOUT MAKING EVEN A SINGLE
CHANGE IN THE QUESTION PLEASEEEEEEEEE!

Rules:
- Keep meaning the same.
- Keeping structure same is not required as long as meaning is not altered.
-DO NOT WRITE THE WORD 'QUESTION'OR 'REWRITTEN QUESTION' WHILE GENERATING PARAPHRASED QUESTIONS
- DO NOT change numbers.
- DO NOT solve the question.
- DO NOT add explanation.
- DO NOT rearrange answer order.

Return ONLY:
MAKE AT LEAST SOME ALTERATION TO "EVERY SINGLE QUESTION". REMEMBER THIS POINT IT IS 
VERY IMPORTANT REREAD THIS POINT BEFORE YOU PARAPHRASE EVERY SINGLE QUESTION.
IMPORTANT:DO NOT CHANGE SYMBOLS SUCH AS UNION OR IMPLIES OR LOGICAL NOT,ETC EVERRRRR
ALSO DONT CHANGE NUMBERS AND FRACTIONS IN QUESTIONS EVERRRRR!!!!

A. <rewritten option>
B. <rewritten option>
C. <rewritten option>
D. <rewritten option>

{question}

A. {choices[0]}
B. {choices[1]}
C. {choices[2]}
D. {choices[3]}
"""

    messages = [
        {"role": "system", "content": "You are a careful academic editor who lightly paraphrases questions."},
        {"role": "user", "content": user_prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=220,
        temperature=0.25,
        top_p=0.85,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    input_length = inputs["input_ids"].shape[-1]
    generated_tokens = outputs[0][input_length:]

    text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    return text

In [6]:
import re

def extract_question_and_choices(text):

    lines = [l.strip() for l in text.split("\n") if l.strip()]

    question_lines = []
    choices = []

    for line in lines:
        if re.match(r"^[A-D]\.", line):
            choices.append(line[2:].strip())
        else:
            question_lines.append(line)

    question = " ".join(question_lines).strip()

    if len(choices) != 4:
        return None, None

    return question, choices

In [7]:
results = []
model_success_count = 0

for ex in tqdm(data):

    example_id = ex["id"]
    subject = ex["subject"]
    question = ex["question"]
    choices = ex["choices"]
    answer = ex["answer"]

    success = False
    retry = 0
    current_threshold = START_SIM

    while retry < MAX_RETRIES and current_threshold >= MIN_SIM:

        paraphrased_text = paraphrase_question(question, choices)

        new_question, new_choices = extract_question_and_choices(paraphrased_text)

        if new_question is None:
            retry += 1
            current_threshold -= STEP
            continue

        similarity = semantic_similarity(question, new_question)

        # Accept if within allowed range
        if current_threshold <= similarity < MAX_SIM:
            success = True
            break

        retry += 1
        current_threshold -= STEP

    if not success:
        new_question = question
        new_choices = choices
    else:
        model_success_count += 1

    print("\n====================================")
    print("ORIGINAL:")
    print(question)
    print("\nPARAPHRASED:")
    print(new_question)
    print("SIMILARITY:", similarity if success else "FAILED")
    print("THRESHOLD USED:", current_threshold if success else "NONE")
    print("====================================\n")

    results.append({
        "id": example_id,
        "subject": subject,
        "question": new_question,
        "choices": new_choices,
        "answer": answer
    })

with open(OUTPUT_FILE, "w") as f:
    json.dump(results, f, indent=2)

print("\n==============================")
print("Total Questions:", len(data))
print("Successful Paraphrases:", model_success_count)
print("Fallback to Original:", len(data) - model_success_count)
print("==============================")

  0%|          | 1/200 [00:13<45:10, 13.62s/it]


ORIGINAL:
Which of the following best accounts for the negative slope of the liquid-solid equilibrium line in the phase diagram for water?

PARAPHRASED:
Which of the following best explains the negative slope of the liquid-solid equilibrium line in the phase diagram for water?
SIMILARITY: 0.9813383221626282
THRESHOLD USED: 0.9



  1%|          | 2/200 [00:19<29:51,  9.05s/it]


ORIGINAL:
Which of the following policies is most likely to bring about economic growth in the long run?

PARAPHRASED:
Which of the following policies is most likely to foster economic growth in the long run?
SIMILARITY: 0.9791312217712402
THRESHOLD USED: 0.9



  2%|▏         | 3/200 [00:24<23:44,  7.23s/it]

...


ORIGINAL:
Epictetus claims that when someone strikes you, what really angers you is:

PARAPHRASED:
Epictetus asserts that when someone hits you, what truly provokes you is:
SIMILARITY: 0.8268426656723022
THRESHOLD USED: 0.7999999999999999



100%|█████████▉| 199/200 [50:48<00:16, 16.13s/it]


ORIGINAL:
Which of the following is used to remove the soil enclosing site materials?

PARAPHRASED:
Which of the following tools is used to remove the soil surrounding the site materials?
SIMILARITY: 0.9176969528198242
THRESHOLD USED: 0.7999999999999999



100%|██████████| 200/200 [50:58<00:00, 15.29s/it]


ORIGINAL:
What was the name of the 1999 art exhibit that sparked a national debate about censorship?

PARAPHRASED:
What was the title of the 1999 art exhibit that sparked a national debate about censorship?
SIMILARITY: 0.9924136400222778
THRESHOLD USED: 0.85


Total Questions: 200
Successful Paraphrases: 189
Fallback to Original: 11
